# 用 TensorFlow 做 NN3 預測台股超額報酬：教學筆記

這份筆記照 Choi, Jiang & Zhang（2025）〈Alpha Go Everywhere〉的方法，用 TEJ 台股資料訓練一個三層神經網路，最後看佳必琪（6197）的預測效果。

流程分五步：資料 → 前處理 → 建模 → 擴張式重訓 → 評估。完整的訓練程式在 `nn_tf.py`，評估程式在 `evaluate.py`，這份筆記只是逐步解說。

## 1. 資料與目標
- **y**：下個月的含息報酬，減掉市場報酬。市場報酬用樣本的市值加權平均當代理。
  - 為什麼要減市場？我們要學的是「哪檔股票會比大家好」，也就是橫斷面的差異，不是去猜大盤會漲還是跌。
- **X**：14 個價量特徵，都只用 t 月以前的資料算：
  - 動能：mom_1, mom_6, mom_12, mom_36, chmom_6
  - 波動：retvol, maxret, beta
  - 規模與流動性：mvel1, dolvol, turn, stdturn, stddolvol, ill

In [1]:
import numpy as np, pandas as pd
from nn_tf import load_panel, build_nn3, fit_ensemble, FEAT
P = load_panel()   # 讀 TEJ panel，並做縮尾、減市場、每月 z-score
P[['mon','code','y']+FEAT].head()

I0000 00:00:1793246711.682238  905093 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793246711.729450  905093 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1793246712.629641  905093 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


,mon,code,y,mom_1,mom_6,mom_12,mom_36,chmom_6,retvol,maxret,mvel1,dolvol,turn,stdturn,stddolvol,ill,beta
0,2001-09-30,1101,-0.125903,-0.168469,-0.428069,-0.613210,0.0,0.176469,-0.896229,-0.939261,1.470088,0.316926,-0.448035,-0.587962,-0.286808,-0.437638,0.0
1,2001-09-30,1102,-0.217959,0.772416,0.429230,0.276719,0.0,0.366382,-1.029096,-0.936218,1.587747,-0.161805,-0.650367,-0.753757,-0.369117,-0.370115,0.0
2,2001-09-30,1103,-0.041709,0.735705,0.748935,0.548851,0.0,0.470491,-1.513568,-1.346445,0.500984,-0.697902,-0.708795,-0.842708,-1.278358,-0.229467,0.0
3,2001-09-30,1104,-0.151574,1.911895,1.417648,1.302498,0.0,0.939912,-0.407256,0.105517,0.156614,-0.121606,-0.320529,-0.582561,0.688876,0.184480,0.0
4,2001-09-30,1107,-0.183948,-0.898829,0.233700,-0.078195,0.0,0.128322,1.699165,2.126310,-0.459073,-0.001270,-0.546688,-0.631915,0.160357,-0.339355,0.0


## 2. 前處理（在 `load_panel` 裡）
1. 每月把報酬上下 2.5% 縮尾，避免極端值主導訓練。
2. 每個特徵每月先在 1%/99% 縮尾，再做 z-score（平均 0、標準差 1）。
   - 這樣每個月的特徵尺度都一樣。
   - 只用同一個月的橫斷面資料，所以沒有 look-ahead。
3. 缺值補 0，等於補成當月平均。

## 3. 切分：30% 訓練 / 20% 驗證 / 50% 測試
- 依時間順序切，不能隨機切。隨機切的話，模型會偷看到未來。
- 訓練：2001/09–2009/02
- 驗證：2009/03–2014/02
- 測試：2014/03–2026/08
- 每過 12 個月重訓一次。訓練期逐年加長（expanding），驗證期固定 60 個月，跟著往前滾。

In [2]:
months = sorted(P['mon'].unique()); T = len(months)
ntr, nva = round(.3*T), round(.2*T)
print('訓練期到', months[ntr-1].date(), '｜驗證期到', months[ntr+nva-1].date(), '｜測試從', months[ntr+nva].date())

訓練期到 2009-02-28 ｜驗證期到 2014-02-28 ｜測試從 2014-03-31


## 4. 模型：NN3（32-16-8）
- 每層都是 Dense → BatchNormalization → ReLU。
  - BatchNorm 讓每層的輸入分布比較穩定，訓練比較快。
  - ReLU 是非線性函數，讓網路可以學到特徵之間的交互作用。
- **L1 正則化**：懲罰權重的絕對值，逼不重要的權重變成 0，可以減少過擬合。強度在驗證集上從 1e-5、1e-4、1e-3 裡挑。
- **Early stopping**：驗證集的 loss 連續 5 個 epoch 沒有改善就停下來，並回到最好的那組權重。
- **5 個 seed 平均（ensemble）**：神經網路每次的初始權重不同，結果會跳動，平均幾個可以降低這種雜訊。

In [3]:
m = build_nn3(len(FEAT)); m.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 14)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8)              │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,377 (5.38 KB)

 Trainable params: 1,265 (4.94 KB)

 Non-trainable params: 112 (448.00 B)

## 5. 示範：只跑第一個測試年
跑完整 13 年要用 `python nn_tf.py`。這裡只示範一次，而且只用一個 L1 值。

In [4]:
k = ntr+nva; tr, va, te = months[:k-nva], months[k-nva:k], months[k:k+12]
get = lambda s: (P[P.mon.isin(s)][FEAT].values.astype('float32'), P[P.mon.isin(s)]['y'].values.astype('float32'))
(Xtr,ytr),(Xva,yva),(Xte,yte) = get(tr), get(va), get(te)
f = fit_ensemble(Xtr, ytr, Xva, yva, l1=1e-4, seeds=2)
pred = f(Xte)
print('樣本外 R2 =', 1 - ((yte-pred)**2).sum() / (yte**2).sum())

樣本外 R2 = -1.7642975e-05


## 6. 評估（`evaluate.py`）
- **樣本外 R²**：分母用沒有去平均的 y²，照 GKX 的作法。月報酬本來就很難預測，所以 R² 接近 0 甚至是負的很正常。
- **十分組多空**：每月依預測值把股票分成 10 組，做多最高組、放空最低組，算年化 Sharpe。這是論文的主要指標。
- **6197 擇時**：預測超額 > 0 就持有 6197，否則持有市場；另外算一版扣掉手續費和證交稅的。

In [5]:
pd.read_csv('out/6197_metrics.csv')

,模型,方向命中率%,持有月數,交易次數,年化報酬%,Sharpe,最大回撤%,累積報酬%,勝過市場月%,扣成本_年化報酬%,扣成本_Sharpe,扣成本_最大回撤%,扣成本_累積報酬%,扣成本_勝過市場月%
0,OLS-3,52.00,131.0,4.0,25.13,0.81,-39.85,1176.74,45.33,25.04,0.81,-40.21,1161.36,45.33
1,OLS,55.33,86.0,30.0,28.63,1.18,-20.89,2378.69,32.00,27.93,1.15,-21.94,2169.97,32.00
2,ENet,51.33,90.0,24.0,25.93,1.05,-27.55,1654.81,31.33,25.37,1.02,-28.53,1535.37,31.33
3,RF,60.67,78.0,14.0,31.51,1.29,-20.41,3403.34,32.00,31.18,1.28,-20.41,3265.26,32.00
4,GBRT+H,59.33,94.0,26.0,32.08,1.11,-21.26,3213.78,36.67,31.48,1.09,-21.62,2974.54,36.67
5,NN1,57.33,95.0,30.0,30.29,1.17,-20.74,2776.63,36.00,29.59,1.14,-21.80,2534.85,36.00
6,NN2,56.00,95.0,32.0,29.14,1.14,-29.22,2427.32,35.33,28.40,1.11,-30.16,2200.79,35.33
7,NN3,62.67,83.0,36.0,34.07,1.37,-19.84,4621.60,34.67,33.23,1.33,-20.18,4151.09,34.00
8,TF-NN3,57.33,69.0,30.0,31.96,1.30,-22.24,3570.80,27.33,31.26,1.27,-22.24,3268.37,27.33
9,買進持有6197,NaN,150.0,2.0,28.78,0.81,-38.05,1621.48,51.33,28.73,0.81,-38.05,1611.67,51.33


## 7. 限制
- 只用一檔股票、約 150 個月，樣本很小，結果的雜訊很大。
- 論文的 36 個特徵這裡只有 12 個，另外 2 個是自己加的（mom_36、beta）。
- 市場報酬是用樣本市值加權平均代理的。
- 有扣手續費和證交稅，但沒有考慮滑價和流動性。